# Matryoshka Sparse Autoencoder for AlphaFold2 Pair Representations

**Adapted from "Scaling SAEs to ESM2" methodology**

## Key Design Principles

1. **CRITICAL FIX**: Each residue pair (i, j) is treated as a single 'token' of dimension **128**.
   - Input is reshaped from `[Batch, L, L, 128]` → `[N_pairs, 128]` where `N_pairs = B * L * L`
   - This learns features of **PAIRS** (hydrogen bonds, hydrophobic contacts, salt bridges) rather than memorizing specific protein layouts.

2. **TopK Activation**: Uses `TopK(k=64)` instead of L1 penalty for sparsity (sparsity budget).
   - More stable training, easier hyperparameter tuning.
   - Favored in recent scaling studies (Anthropic, EleutherAI).

3. **Matryoshka Nested Reconstruction**: The latent space has nested structure `[128, 1024, 2048, 4096]`.
   - First 128 latents should be most important, next 896 (up to 1024) next important, etc.
   - Enables adaptive compute: use fewer latents for fast inference, more for accuracy.

4. **Standard SAE Formulation**:
   - Encoder: `z = TopK(W_enc @ (x - b_dec) + b_enc)`
   - Decoder: `x_hat = W_dec @ z + b_dec`
   - Decoder weights normalized to unit norm after each step.

## Setup for GPU cluster (run this first)

**Folder layout:** Put everything in the **same directory** as this notebook:
- `Proteins_layer47/` or `protein_layer47/` — pair representation `.npy` files (input data)

Reconstructed pair representations are written to **`Arisa_reconstructed/`** (e.g. `7b3a_A_pair_block_47_reconstructed.npy`).

Then run the cell below to set the working directory and verify paths. On a cluster, set `PROJECT_DIR` if your job starts in a different folder.

In [ ]:
# Set working directory so all paths are relative to the notebook (for GPU cluster runs)
import os
from pathlib import Path

PROJECT_DIR = Path("/storage/ice1/1/0/varisa3/AlphaFold_Autoencoder/VizFoldAutoencoder")   # ICE cluster path
if PROJECT_DIR is not None:
    PROJECT_DIR = Path(PROJECT_DIR)
else:
    PROJECT_DIR = Path.cwd()
    # If we're in parent dir (e.g. Autoencoder), step into VizFoldAutoencoder
    if (PROJECT_DIR / "VizFoldAutoencoder" / "matryoshka_sae.ipynb").exists():
        PROJECT_DIR = PROJECT_DIR / "VizFoldAutoencoder"
    elif not (PROJECT_DIR / "matryoshka_sae.ipynb").exists():
        raise FileNotFoundError("Run this notebook from the folder that contains matryoshka_sae.ipynb (or set PROJECT_DIR).")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Contents:", [x for x in os.listdir(".") if not x.startswith(".")][:20])

## 1. Imports and Configuration

In [4]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from datetime import datetime
from typing import Tuple, Dict, List, Optional

# Import from our modules
from af2_autoencoder import (
    PairRepresentationDataset,
    PairChannelNormalizer,
    plot_reconstruction_comparison,
    plot_training_history,
    plot_difference_heatmap,
)

from matryoshka_sae import (
    MatryoshkaSAEConfig,
    MatryoshkaSAE,
    MatryoshkaSAETrainer,
    PairSAEInference,
    load_pair_representations,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

ModuleNotFoundError: No module named 'numpy'

## 2. Device and Training Configuration

In [ ]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Training hyperparameters (all paths relative to working directory set above)
_data_dir = "Proteins_layer47"
for _candidate in ["proteins_layer47", "protein_layer47", "Proteins_layer47"]:
    if Path(_candidate).exists() and Path(_candidate).is_dir():
        _data_dir = _candidate
        break
config = {
    # Data: pair rep .npy directory (auto-detects protein_layer47 or Proteins_layer47)
    "data_dir": _data_dir,
    "output_dir": "output_matryoshka_sae",
    
    # Model architecture
    "input_dim": 128,           # Feature dim per pair (FIXED for AF2)
    "n_latents": 4096,          # Total latent features (32x expansion)
    "topk": 64,                 # Sparsity budget: active latents per sample
    "matryoshka_dims": (128, 1024, 2048, 4096),  # Nested checkpoints
    
    # Training
    "num_epochs": 100,
    "learning_rate": 1e-4,
    "print_every": 10,
    "early_stopping_patience": 50,
    "checkpoint_epochs": [50, 100],  # Save model at these epochs for TM-score comparison
    
    # Train/test split
    "train_split": 0.5,        # 50% train, 50% test
    "split_seed": 42,          # Random seed for reproducible splits
    
    # Normalization
    "use_normalizer": True,
    
    # Save
    "save_model": True,
}

# Create output directory
os.makedirs(config["output_dir"], exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("\nConfiguration:")
for k, v in config.items():
    print(f"  {k}: {v}")

## 3. Load Pair Representations & Train/Test Split

Each protein has shape `(L, L, 128)` where L is the number of residues.

**Key insight**: Instead of flattening to `L*L*128` (massive!), we treat each (i,j) pair as an independent sample of dimension 128.

**Train/test split**: We randomly split the proteins 50/50 — train on half, evaluate on the held-out half to measure generalisation.

In [ ]:
print("=" * 60)
print("STEP 1: Load Pair Representations")
print("=" * 60)

pair_tensors, npy_files = load_pair_representations(config["data_dir"], return_file_paths=True)

print(f"\nLoaded {len(pair_tensors)} proteins")
print(f"Shapes: {[t.shape for t in pair_tensors[:5]]}{'...' if len(pair_tensors) > 5 else ''}")
total_pairs = sum(t.shape[0] * t.shape[1] for t in pair_tensors)
print(f"Total pairs: {total_pairs:,}")

# --- 50/50 Train/Test Split ---
print("\n" + "=" * 60)
print("STEP 1b: Train/Test Split")
print("=" * 60)

n_total = len(pair_tensors)
n_train = int(n_total * config["train_split"])

rng = np.random.RandomState(config["split_seed"])
split_indices = np.arange(n_total)
rng.shuffle(split_indices)

train_indices = sorted(split_indices[:n_train])
test_indices = sorted(split_indices[n_train:])

train_tensors = [pair_tensors[i] for i in train_indices]
test_tensors = [pair_tensors[i] for i in test_indices]
train_files = [npy_files[i] for i in train_indices]
test_files = [npy_files[i] for i in test_indices]

train_pairs = sum(t.shape[0] * t.shape[1] for t in train_tensors)
test_pairs = sum(t.shape[0] * t.shape[1] for t in test_tensors)

print(f"  Total proteins: {n_total}")
print(f"  Train: {len(train_tensors)} proteins ({train_pairs:,} pairs)")
print(f"  Test:  {len(test_tensors)} proteins ({test_pairs:,} pairs)")
print(f"  Split ratio: {config['train_split']:.0%} train / {1-config['train_split']:.0%} test")
print(f"  Random seed: {config['split_seed']}")

# Save split info for reproducibility
split_info_path = os.path.join(config["output_dir"], f"train_test_split_{timestamp}.txt")
with open(split_info_path, "w") as f:
    f.write(f"seed={config['split_seed']}\n")
    f.write(f"train_split={config['train_split']}\n")
    f.write(f"n_total={n_total}\n")
    f.write(f"n_train={len(train_tensors)}\n")
    f.write(f"n_test={len(test_tensors)}\n\n")
    f.write("TRAIN FILES:\n")
    for fp in train_files:
        f.write(f"  {fp.name}\n")
    f.write("\nTEST FILES:\n")
    for fp in test_files:
        f.write(f"  {fp.name}\n")
print(f"  Split info saved to: {split_info_path}")

## 4. Fit Channel Normalizer (Optional but Recommended)

In [ ]:
normalizer = None

if config["use_normalizer"]:
    print("=" * 60)
    print("STEP 2: Fit Channel Normalizer")
    print("=" * 60)
    
    raw_dataset = PairRepresentationDataset(config["data_dir"])
    normalizer = PairChannelNormalizer()
    normalizer.fit(raw_dataset)
else:
    print("Skipping normalization")

## 5. Create Matryoshka SAE Model

The model architecture:
- **Encoder**: `z_pre = W_enc @ (x - b_dec) + b_enc`, then `z = TopK(ReLU(z_pre))`
- **Decoder**: `x_hat = W_dec @ z + b_dec`
- Decoder columns are normalized to unit norm after each gradient step

In [ ]:
print("=" * 60)
print("STEP 3: Create Matryoshka SAE")
print("=" * 60)

sae_config = MatryoshkaSAEConfig(
    input_dim=config["input_dim"],
    n_latents=config["n_latents"],
    matryoshka_dims=config["matryoshka_dims"],
    topk=config["topk"],
    learning_rate=config["learning_rate"],
)

model = MatryoshkaSAE(sae_config)

print(f"\nModel Configuration:")
print(f"  Input dimension: {sae_config.input_dim}")
print(f"  Latent dimensions: {sae_config.n_latents}")
print(f"  Expansion factor: {sae_config.n_latents // sae_config.input_dim}x")
print(f"  TopK sparsity: k={sae_config.topk}")
print(f"  Matryoshka dims: {sae_config.matryoshka_dims}")
print(f"  Total parameters: {model.count_parameters():,}")

# Show model structure
print(f"\nWeight shapes:")
print(f"  W_enc: {model.W_enc.shape} (n_latents x input_dim)")
print(f"  W_dec: {model.W_dec.shape} (input_dim x n_latents)")
print(f"  b_enc: {model.b_enc.shape}")
print(f"  b_dec: {model.b_dec.shape}")

## 6. Verify Reshaping Logic

**This is the critical fix**: Let's verify that the model correctly reshapes inputs.

In [ ]:
print("=" * 60)
print("Verifying Reshaping Logic")
print("=" * 60)

# Test with a single protein from the train set
test_protein = train_tensors[0]
L = test_protein.shape[0]

print(f"\nTest protein shape: {test_protein.shape}")
print(f"  This is (L, L, 128) = ({L}, {L}, 128)")
print(f"  Total elements: {test_protein.numel():,}")

# Test the internal flattening
x_flat, original_shape = model._flatten_pairs(test_protein)
print(f"\nAfter _flatten_pairs:")
print(f"  Flattened shape: {x_flat.shape}")
print(f"  This is (N_pairs, 128) = ({L*L}, 128)")
print(f"  Original shape stored: {original_shape}")

# Test unflattening
x_restored = model._unflatten_pairs(x_flat, original_shape)
print(f"\nAfter _unflatten_pairs:")
print(f"  Restored shape: {x_restored.shape}")
print(f"  Shapes match: {x_restored.shape == test_protein.shape}")

# Test with batched input
batch = torch.stack([train_tensors[0], train_tensors[0]])  # Batch of 2
x_flat_batch, orig_shape_batch = model._flatten_pairs(batch)
print(f"\nBatched input: {batch.shape}")
print(f"  Flattened: {x_flat_batch.shape}")
print(f"  Each pair is now an independent sample!")

## 7. Test Forward Pass

In [ ]:
print("=" * 60)
print("Testing Forward Pass")
print("=" * 60)

model.eval()
with torch.no_grad():
    # Forward pass
    x_hat, z = model(test_protein)
    
    print(f"\nInput shape: {test_protein.shape}")
    print(f"Output shape: {x_hat.shape}")
    print(f"Latent shape: {z.shape}")
    
    # Check sparsity
    n_active = (z != 0).sum().item()
    n_total = z.numel()
    sparsity = n_active / n_total
    print(f"\nSparsity check:")
    print(f"  Active entries: {n_active:,} / {n_total:,}")
    print(f"  Sparsity (fraction active): {sparsity:.4f}")
    print(f"  Expected: {config['topk']} / {config['n_latents']} = {config['topk']/config['n_latents']:.4f}")
    
    # Test Matryoshka reconstruction
    print(f"\nMatryoshka reconstructions:")
    x_hat_full, z_full, matryoshka_recons = model(test_protein, return_all_reconstructions=True)
    for dim, recon in matryoshka_recons.items():
        mse = F.mse_loss(recon, test_protein).item()
        print(f"  {dim} latents: MSE = {mse:.6f}")

## 8. Create Trainer and Train

In [ ]:
print("=" * 60)
print("STEP 4: Create Trainer")
print("=" * 60)

trainer = MatryoshkaSAETrainer(
    model=model,
    config=sae_config,
    normalizer=normalizer,
    device=device,
    learning_rate=config["learning_rate"],
)

In [ ]:
print("=" * 60)
print("STEP 5: Train Matryoshka SAE (train set only)")
print("=" * 60)

history = trainer.train(
    pair_tensors=train_tensors,
    num_epochs=config["num_epochs"],
    print_every=config["print_every"],
    lr_schedule=True,
    early_stopping_patience=config["early_stopping_patience"],
    checkpoint_epochs=config["checkpoint_epochs"],
    checkpoint_dir=config["output_dir"],
)

## 9. Evaluate Model (Train & Test Sets)

In [ ]:
# --- Evaluate on TRAIN set ---
print("=" * 60)
print("STEP 6a: Evaluate on TRAIN set")
print("=" * 60)

train_metrics = trainer.evaluate(train_tensors)

print(f"\nTrain Results:")
print(f"  Total loss: {train_metrics['total_loss']:.6f}")
print(f"  Sparsity: {train_metrics['sparsity']:.4f}")
print(f"\nTrain per-dimension losses:")
for dim, loss in train_metrics['per_dim_losses'].items():
    print(f"  {dim:4d} latents: {loss:.6f}")

# --- Evaluate on TEST set ---
print("\n" + "=" * 60)
print("STEP 6b: Evaluate on TEST set")
print("=" * 60)

test_metrics = trainer.evaluate(test_tensors)

print(f"\nTest Results:")
print(f"  Total loss: {test_metrics['total_loss']:.6f}")
print(f"  Sparsity: {test_metrics['sparsity']:.4f}")
print(f"\nTest per-dimension losses:")
for dim, loss in test_metrics['per_dim_losses'].items():
    print(f"  {dim:4d} latents: {loss:.6f}")

# --- Summary comparison ---
print("\n" + "=" * 60)
print("Train vs Test Comparison")
print("=" * 60)
print(f"  {'Metric':<25} {'Train':>10} {'Test':>10} {'Ratio':>10}")
print(f"  {'Total loss':<25} {train_metrics['total_loss']:>10.6f} {test_metrics['total_loss']:>10.6f} {test_metrics['total_loss']/max(train_metrics['total_loss'],1e-10):>10.2f}x")
for dim in config['matryoshka_dims']:
    tr = train_metrics['per_dim_losses'][dim]
    te = test_metrics['per_dim_losses'][dim]
    print(f"  {f'd={dim} loss':<25} {tr:>10.6f} {te:>10.6f} {te/max(tr,1e-10):>10.2f}x")

print(f"\nPer-protein losses (TEST set):")
for idx, loss in sorted(test_metrics['per_protein_losses'].items(), key=lambda x: x[1]):
    L = test_tensors[idx].shape[0]
    print(f"  Test protein {idx} (L={L}): {loss:.6f}")

## 10. Visualize Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Total loss
axes[0].plot(history['epoch'], history['total_loss'], 'b-', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Total Matryoshka Loss')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Per-dimension losses
colors = plt.cm.viridis(np.linspace(0, 1, len(config['matryoshka_dims'])))
for (dim, losses), color in zip(history['matryoshka_losses'].items(), colors):
    axes[1].plot(history['epoch'], losses, label=f'{dim} latents', color=color, linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].set_title('Matryoshka Dimension Losses')
axes[1].set_yscale('log')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Sparsity
axes[2].plot(history['epoch'], history['sparsity'], 'g-', linewidth=2)
axes[2].axhline(y=config['topk']/config['n_latents'], color='r', linestyle='--', label=f'Expected: {config["topk"]}/{config["n_latents"]}')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Sparsity (fraction active)')
axes[2].set_title('Activation Sparsity')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config['output_dir'], f'matryoshka_training_history_{timestamp}.png'), dpi=150)
plt.show()

## 11. Visualize Reconstructions at Different Matryoshka Levels

In [ ]:
# Create inference helper
inference = PairSAEInference(model, normalizer, device)

# Use a protein from the TEST set to visualise generalisation
test_idx = 0
original = test_tensors[test_idx]
L = original.shape[0]

print(f"Reconstructing TEST protein {test_idx} (L={L}) — unseen during training")
print(f"Original shape: {original.shape}")

# Reconstruct at different Matryoshka levels
reconstructions = {}
for n_lat in config['matryoshka_dims']:
    recon = inference.reconstruct_for_structure_module(original, n_latents=n_lat)
    reconstructions[n_lat] = recon
    mse = F.mse_loss(recon, original).item()
    corr = np.corrcoef(recon.numpy().flatten(), original.numpy().flatten())[0, 1]
    print(f"  {n_lat:4d} latents: MSE={mse:.6f}, Correlation={corr:.4f}")

In [ ]:
# Plot reconstructions at each Matryoshka level
channel_to_plot = 0  # Which channel to visualize

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Original
im = axes[0].imshow(original[:, :, channel_to_plot].numpy(), cmap='viridis')
axes[0].set_title(f'Original (Channel {channel_to_plot})')
axes[0].set_xlabel('Residue j')
axes[0].set_ylabel('Residue i')
plt.colorbar(im, ax=axes[0])

# Reconstructions at each level
for idx, (n_lat, recon) in enumerate(reconstructions.items()):
    ax = axes[idx + 1]
    mse = F.mse_loss(recon, original).item()
    im = ax.imshow(recon[:, :, channel_to_plot].numpy(), cmap='viridis')
    ax.set_title(f'{n_lat} latents (MSE={mse:.5f})')
    ax.set_xlabel('Residue j')
    ax.set_ylabel('Residue i')
    plt.colorbar(im, ax=ax)

# Difference (full reconstruction)
full_recon = reconstructions[config['matryoshka_dims'][-1]]
diff = (original - full_recon).abs()
im = axes[5].imshow(diff[:, :, channel_to_plot].numpy(), cmap='hot')
axes[5].set_title('Absolute Difference (Full)')
axes[5].set_xlabel('Residue j')
axes[5].set_ylabel('Residue i')
plt.colorbar(im, ax=axes[5])

plt.suptitle(f'Matryoshka Reconstruction Quality (Channel {channel_to_plot})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config['output_dir'], f'matryoshka_reconstructions_{timestamp}.png'), dpi=150)
plt.show()

## 12. Analyze Active Latents (Interpretability)

In [ ]:
# Get active features for each pair position
indices, values = inference.get_feature_activations(original)

print(f"Active feature analysis:")
print(f"  Indices shape: {indices.shape} (L, L, k)")
print(f"  Values shape: {values.shape} (L, L, k)")

# Which latents are most commonly active?
all_indices = indices.reshape(-1).numpy()
unique, counts = np.unique(all_indices, return_counts=True)
top_latents = sorted(zip(unique, counts), key=lambda x: -x[1])[:20]

print(f"\nTop 20 most frequently activated latents:")
for latent_idx, count in top_latents:
    percent = count / len(all_indices) * 100
    print(f"  Latent {latent_idx}: activated {count} times ({percent:.1f}%)")

## Generate reconstructed & original test set outputs

- **`reconstructed_proteins_layer47/`** — SAE reconstructions of the **test** proteins (unseen during training)
- **`original_proteins_layer47/`** — copies of the original test `.npy` files (for easy side-by-side comparison / TM-score later)

In [ ]:
import shutil

recon_dir = Path("reconstructed_proteins_layer47")
orig_dir = Path("original_proteins_layer47")
recon_dir.mkdir(exist_ok=True)
orig_dir.mkdir(exist_ok=True)

print("=" * 60)
print("Saving test set: reconstructed + original .npy files")
print("=" * 60)

for i, tensor in enumerate(test_tensors):
    stem = test_files[i].stem  # e.g. 7b3a_A_pair_block_47

    # SAE reconstruction
    recon = inference.reconstruct_for_structure_module(tensor)
    np.save(str(recon_dir / f"{stem}.npy"), recon.numpy())

    # Copy the original .npy so both dirs have matching filenames
    shutil.copy2(str(test_files[i]), str(orig_dir / f"{stem}.npy"))

    print(f"  {stem}  →  reconstructed + original saved")

print(f"\n  reconstructed_proteins_layer47/  ({len(test_tensors)} files)")
print(f"  original_proteins_layer47/       ({len(test_tensors)} files)")

## Reconstructions saved

- **`reconstructed_proteins_layer47/`** — SAE-reconstructed test set `.npy` files
- **`original_proteins_layer47/`** — matching original test set `.npy` files

Both directories have the same filenames so you can run the structure module on each and then compare TM-scores.

In [ ]:
print("Test set .npy files ready for structure module + TM-score comparison:")
print(f"  reconstructed_proteins_layer47/  ({len(list(recon_dir.glob('*.npy')))} files)")
print(f"  original_proteins_layer47/       ({len(list(orig_dir.glob('*.npy')))} files)")

In [ ]:
# Visualize which latents are active at each position
# Heatmap of the most active latent at each (i, j)
most_active_latent = indices[:, :, 0].numpy()  # Most active latent (highest value)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Most active latent index
im1 = axes[0].imshow(most_active_latent, cmap='tab20')
axes[0].set_title('Most Active Latent Index per Pair')
axes[0].set_xlabel('Residue j')
axes[0].set_ylabel('Residue i')
plt.colorbar(im1, ax=axes[0])

# Activation strength of most active latent
im2 = axes[1].imshow(values[:, :, 0].numpy(), cmap='viridis')
axes[1].set_title('Activation Strength of Most Active Latent')
axes[1].set_xlabel('Residue j')
axes[1].set_ylabel('Residue i')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.savefig(os.path.join(config['output_dir'], f'active_latents_{timestamp}.png'), dpi=150)
plt.show()

## 13. Latent Ablation Study

Understand what individual latents contribute by ablating them.

In [ ]:
# Ablate the most important latent and see the effect
top_latent_idx = top_latents[0][0]
print(f"Ablating latent {top_latent_idx} (most frequently active)")

# Full reconstruction
full_recon = inference.reconstruct_for_structure_module(original)

# Reconstruction with ablated latent
ablated_recon = inference.ablate_latent(original, top_latent_idx)

# Compute difference
ablation_effect = (full_recon - ablated_recon).abs()

print(f"  Full recon MSE: {F.mse_loss(full_recon, original).item():.6f}")
print(f"  Ablated recon MSE: {F.mse_loss(ablated_recon, original).item():.6f}")
print(f"  Mean ablation effect: {ablation_effect.mean().item():.6f}")

In [ ]:
# Visualize ablation effect
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

channel = 0

im1 = axes[0].imshow(full_recon[:, :, channel].numpy(), cmap='viridis')
axes[0].set_title('Full Reconstruction')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(ablated_recon[:, :, channel].numpy(), cmap='viridis')
axes[1].set_title(f'With Latent {top_latent_idx} Ablated')
plt.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow(ablation_effect[:, :, channel].numpy(), cmap='hot')
axes[2].set_title(f'Ablation Effect (|diff|)')
plt.colorbar(im3, ax=axes[2])

for ax in axes:
    ax.set_xlabel('Residue j')
    ax.set_ylabel('Residue i')

plt.suptitle(f'Effect of Ablating Latent {top_latent_idx} (Channel {channel})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config['output_dir'], f'ablation_latent{top_latent_idx}_{timestamp}.png'), dpi=150)
plt.show()

## 14. Decoder Dictionary Visualization

Each column of W_dec is a "dictionary direction" - the pattern that latent represents in the input space.

In [ ]:
# Get decoder directions for top latents
decoder_dirs = inference.get_all_decoder_directions()  # (128, n_latents)

print(f"Decoder dictionary shape: {decoder_dirs.shape}")
print(f"Each column is a 128-dimensional direction representing a learned feature.")

# Visualize top 10 latent directions
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i, (latent_idx, count) in enumerate(top_latents[:10]):
    direction = decoder_dirs[:, latent_idx].numpy()
    axes[i].bar(range(128), direction, width=1.0, color='steelblue')
    axes[i].set_title(f'Latent {latent_idx}\n(active {count} times)')
    axes[i].set_xlabel('Channel')
    axes[i].set_ylabel('Weight')
    axes[i].set_xlim(0, 128)

plt.suptitle('Decoder Directions for Most Active Latents', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config['output_dir'], f'decoder_directions_{timestamp}.png'), dpi=150)
plt.show()

## 15. Save Model

In [ ]:
if config['save_model']:
    print("=" * 60)
    print("STEP 7: Save Model")
    print("=" * 60)
    
    model_path = os.path.join(config['output_dir'], f'matryoshka_sae_{timestamp}.pt')
    trainer.save(model_path)
    
    if normalizer is not None:
        normalizer_path = os.path.join(config['output_dir'], f'normalizer_{timestamp}.pt')
        normalizer.save(normalizer_path)
        print(f"Normalizer saved to: {normalizer_path}")
    
    print(f"\nSaved files:")
    for f in os.listdir(config['output_dir']):
        if timestamp in f:
            fpath = os.path.join(config['output_dir'], f)
            size = os.path.getsize(fpath) / 1024
            print(f"  {f}: {size:.1f} KB")

## 16. Summary and Next Steps

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"""
Matryoshka SAE Training Complete!

Model Architecture:
  - Input dimension: {config['input_dim']} (per pair)
  - Latent dimensions: {config['n_latents']} ({config['n_latents']//config['input_dim']}x expansion)
  - TopK sparsity: k={config['topk']}
  - Matryoshka levels: {config['matryoshka_dims']}
  - Parameters: {model.count_parameters():,}

Train/Test Split:
  - Total proteins: {len(train_tensors) + len(test_tensors)}
  - Train: {len(train_tensors)} proteins
  - Test:  {len(test_tensors)} proteins
  - Split: {config['train_split']:.0%} / {1-config['train_split']:.0%}

Training Results (train set):
  - Train loss: {train_metrics['total_loss']:.6f}
  - Train sparsity: {train_metrics['sparsity']:.4f}
  - Epochs trained: {len(history['epoch'])}

Generalisation Results (test set):
  - Test loss:  {test_metrics['total_loss']:.6f}
  - Test sparsity: {test_metrics['sparsity']:.4f}
  - Test/Train loss ratio: {test_metrics['total_loss']/max(train_metrics['total_loss'],1e-10):.2f}x

Matryoshka Performance (Train / Test):
""")
for dim in config['matryoshka_dims']:
    tr = train_metrics['per_dim_losses'][dim]
    te = test_metrics['per_dim_losses'][dim]
    print(f"  {dim:4d} latents: Train MSE = {tr:.6f}  |  Test MSE = {te:.6f}")

print(f"""
Output Directories:
  - reconstructed_proteins_layer47/  (SAE reconstructions of test proteins)
  - original_proteins_layer47/       (original test proteins for comparison)

Next Steps:
  1. Run structure module on both dirs to get PDBs
  2. Run compute_tm_scores_sae_pdb.py to compare TM-scores
  3. Compare test loss to train loss — ratio close to 1.0 means good generalisation
  4. Analyze which latents correspond to known structural features
""")

---

## Appendix: Loading a Saved Model

In [ ]:
# Example: How to load a saved model
# 
# from matryoshka_sae import MatryoshkaSAETrainer
# from af2_autoencoder import PairChannelNormalizer
# 
# # Load normalizer
# normalizer = PairChannelNormalizer.load("output_matryoshka_sae/normalizer_XXXXXX.pt")
# 
# # Load trainer (includes model)
# trainer = MatryoshkaSAETrainer.load(
#     "output_matryoshka_sae/matryoshka_sae_XXXXXX.pt",
#     normalizer=normalizer,
#     device="cuda"
# )
# 
# # Use for inference
# inference = PairSAEInference(trainer.model, normalizer)
# recon = inference.reconstruct_for_structure_module(pair_rep)